# Random-Label Control Table

Compact paper and audit tables for structured-random attentive controls. Main-task references use the current recovered seed-42 layerwise source rows only, never seed averages. The paper table selects the main-task layer by seed-42 test primary metric and reports random-label primary-metric and accuracy performance at that same layer.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Optional

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = Path.cwd() if start is None else start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'run.py').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Could not find Probe4Physics repo root.')


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / 'results'
MAIN_CONFIG_CSV = RESULTS_DIR / 'verified_layerwise_probe_configs.csv'
ARTIFACT_ROOTS = [
    REPO_ROOT / 'artifacts' / 'probes',
    Path('/scratch-shared/spunzo1/probe4physics/artifacts/probes'),
    Path('/gpfs/scratch1/shared/spunzo1/probe4physics/artifacts/probes'),
]

DATASETS = [
    {'key': 'intphys2', 'label': 'IntPhys2', 'metric': 'voe_accuracy', 'epochs': 90, 'patience': 20},
    {'key': 'mvp', 'label': 'MVP', 'metric': 'pair_consistency', 'epochs': 30, 'patience': 5},
]

MODELS = [
    {
        'label': 'V-JEPA',
        'slug': 'jepa_v1_vith16_384',
        'main_model': 'V-JEPA',
        'main_backbone': 'ViT-H/16',
        'main_experiment': 'main',
    },
    {
        'label': 'V-JEPA 2',
        'slug': 'jepa_v2_vitg_384',
        'main_model': 'V-JEPA 2',
        'main_backbone': 'ViT-G/16',
        'main_experiment': 'main',
    },
    {
        'label': 'V-JEPA 2.1',
        'slug': 'jepa_v2_1_vitG_384',
        'main_model': 'V-JEPA 2.1',
        'main_backbone': 'ViT-Gigantic/16',
        'main_experiment': 'main',
    },
    {
        'label': 'VideoMAE',
        'slug': 'videomae_vit_huge_16_224',
        'main_model': 'VideoMAE',
        'main_backbone': 'ViT-H/16',
        'main_experiment': 'main',
    },
    {
        'label': 'VideoMAE-v2',
        'slug': 'videomae_v2_vit_giant_16_224',
        'main_model': 'VideoMAE-v2',
        'main_backbone': 'ViT-G/16',
        'main_experiment': 'main',
    },
    {
        'label': 'LTX-Video',
        'slug_by_dataset': {
            'intphys2': 'ltx_video_ltxv_13b_0_9_8_distilled',
            'mvp': 'ltx_video_ltxv_13b_0_9_8_distilled',
        },
        'main_model': 'LTX-Video',
        'main_backbone': 'LTX-13B',
        'main_experiment': 'ltx',
    },
]

LTX_LABEL_TO_SLOT = {
    'noise_0.6_block_24': '18',
    'noise_0.5_block_24': '22',
    'noise_0.4_block_24': '26',
    'noise_0.2_block_24': '34',
    'noise_0.1_block_24': '38',
    'noise_0.2_block_48': '36',
    'noise_0.1_block_36': '39',
}

LTX_SLOT_TO_LABEL = {slot: label for label, slot in LTX_LABEL_TO_SLOT.items()}


def format_layer_display(model: dict, layer: object, layer_label: object = '') -> str:
    normalized_layer = norm_layer(layer)
    if model.get('main_experiment') != 'ltx':
        return normalized_layer or 'NA'

    labels = [norm_layer(layer_label), normalized_layer]
    if normalized_layer in LTX_SLOT_TO_LABEL:
        labels.append(LTX_SLOT_TO_LABEL[normalized_layer])

    for label in labels:
        if label.startswith('noise_') and '_block_' in label:
            noise, block = label.removeprefix('noise_').split('_block_', 1)
            return f'{block} (noise {noise})'
    return normalized_layer or norm_layer(layer_label) or 'NA'


In [2]:
def model_slug(model: dict, dataset_key: str) -> str:
    return model.get('slug_by_dataset', {}).get(dataset_key, model.get('slug', ''))


def expected_group_name(dataset: dict, model: dict) -> str:
    slug = model_slug(model, dataset['key'])
    return (
        f"{dataset['key']}_probe_temporal_attn_{slug}_"
        f"structured_random_lr_matrix_ep{dataset['epochs']}_pat{dataset['patience']}"
    )


def candidate_group_dirs(dataset: dict, model: dict) -> list[Path]:
    group = expected_group_name(dataset, model)
    return [root / dataset['key'] / group for root in ARTIFACT_ROOTS]


def first_existing(paths: list[Path]) -> Optional[Path]:
    for path in paths:
        if path.exists():
            return path
    return None


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def as_float(value: object) -> Optional[float]:
    try:
        if value in (None, ''):
            return None
        number = float(value)
    except (TypeError, ValueError):
        return None
    if pd is not None and pd.isna(number):
        return None
    return number


def fmt(value: object) -> str:
    number = as_float(value)
    if number is None:
        return 'NA'
    return f'{number:.2f}'


def norm_layer(value: object) -> str:
    if value is None:
        return ''
    if pd is not None and pd.isna(value):
        return ''
    number = as_float(value)
    if number is not None:
        return str(int(number)) if float(number).is_integer() else str(number)
    return str(value).strip()


def split_metric(layer: dict, split: str, name: str) -> Optional[float]:
    metrics = layer.get('eval', {}).get('metrics_by_split', {}).get(split, {})
    if not isinstance(metrics, dict):
        return None
    return as_float(metrics.get(name))


def random_layers(dataset: dict, model: dict) -> tuple[list[dict], str]:
    group_dir = first_existing(candidate_group_dirs(dataset, model))
    if group_dir is None:
        return [], 'missing group'

    summary_json = group_dir / 'train_eval_summary.json'
    if summary_json.exists():
        summary = read_json(summary_json)
        return [layer for layer in summary.get('layers', []) if isinstance(layer, dict)], 'done'

    summary_csv = group_dir / 'train_eval_summary.csv'
    if not summary_csv.exists():
        return [], 'running/no summary'
    return read_csv_rows(summary_csv), 'done'


def layer_candidates(layer: object, layer_label: object) -> set[str]:
    candidates = {norm_layer(layer), norm_layer(layer_label)}
    for value in list(candidates):
        mapped = LTX_LABEL_TO_SLOT.get(value)
        if mapped:
            candidates.add(mapped)
    candidates.discard('')
    return candidates


def random_layer_metrics(dataset: dict, model: dict, layer: object, layer_label: object) -> dict[str, object]:
    layers, status = random_layers(dataset, model)
    base = {'train': None, 'val': None, 'test': None, 'test_acc': None, 'lr': 'NA', 'status': status}
    if not layers:
        return base

    candidates = layer_candidates(layer, layer_label)
    for item in layers:
        item_candidates = layer_candidates(item.get('layer'), item.get('layer_label'))
        if candidates & item_candidates:
            metric = dataset['metric']
            if 'eval' in item:
                return {
                    'train': split_metric(item, 'train', metric),
                    'val': split_metric(item, 'val', metric),
                    'test': split_metric(item, 'test', metric),
                    'test_acc': split_metric(item, 'test', 'accuracy'),
                    'lr': str(item.get('learning_rate_tag') or item.get('learning_rate') or 'NA'),
                    'status': 'done',
                }
            return {
                'train': as_float(item.get(f'train_{metric}')),
                'val': as_float(item.get(f'val_{metric}') or item.get('selection_metric')),
                'test': as_float(item.get(f'test_{metric}') or item.get('objective_metric')),
                'test_acc': as_float(item.get('test_accuracy')),
                'lr': item.get('selected_lr_tag') or item.get('selected_lr') or 'NA',
                'status': 'done',
            }
    return {**base, 'status': f'missing same layer: {sorted(candidates)}'}


def load_control_row(dataset: dict, model: dict) -> dict[str, str]:
    base = {
        'Dataset': dataset['label'],
        'Model': model['label'],
        'Best layer': 'NA',
        'LR': 'NA',
        'Train VOE/PC': 'NA',
        'Train acc': 'NA',
        'Val VOE/PC': 'NA',
        'Val acc': 'NA',
        'Test VOE/PC': 'NA',
        'Test acc': 'NA',
        'Status': 'NA',
    }
    layers, status = random_layers(dataset, model)
    if not layers:
        return {**base, 'Status': status}

    metric = dataset['metric']
    if 'eval' in layers[0]:
        best = max(layers, key=lambda layer: split_metric(layer, 'val', metric) or float('-inf'))
        return {
            **base,
            'Best layer': format_layer_display(model, best.get('layer'), best.get('layer_label')),
            'LR': str(best.get('learning_rate_tag') or best.get('learning_rate') or 'NA'),
            'Train VOE/PC': fmt(split_metric(best, 'train', metric)),
            'Train acc': fmt(split_metric(best, 'train', 'accuracy')),
            'Val VOE/PC': fmt(split_metric(best, 'val', metric)),
            'Val acc': fmt(split_metric(best, 'val', 'accuracy')),
            'Test VOE/PC': fmt(split_metric(best, 'test', metric)),
            'Test acc': fmt(split_metric(best, 'test', 'accuracy')),
            'Status': 'done',
        }

    best = max(layers, key=lambda row: as_float(row.get('selection_metric') or row.get(f'val_{metric}')) or float('-inf'))
    return {
        **base,
        'Best layer': format_layer_display(model, best.get('layer'), best.get('layer_label')),
        'LR': best.get('selected_lr_tag') or best.get('selected_lr') or 'NA',
        'Train VOE/PC': fmt(best.get(f'train_{metric}')),
        'Train acc': fmt(best.get('train_accuracy')),
        'Val VOE/PC': fmt(best.get(f'val_{metric}') or best.get('selection_metric')),
        'Val acc': fmt(best.get('val_accuracy')),
        'Test VOE/PC': fmt(best.get(f'test_{metric}') or best.get('objective_metric')),
        'Test acc': fmt(best.get('test_accuracy')),
        'Status': 'done',
    }


def main_seed42_row(dataset: dict, model: dict) -> dict[str, object]:
    if pd is None:
        raise ModuleNotFoundError('pandas is required to load the current seed-42 main-task source table.')
    if not MAIN_CONFIG_CSV.exists():
        return {'status': f'missing {MAIN_CONFIG_CSV}'}

    main = pd.read_csv(MAIN_CONFIG_CSV)
    subset = main[
        main['dataset'].astype(str).eq(dataset['label'])
        & main['experiment'].astype(str).eq(model['main_experiment'])
        & main['model'].astype(str).eq(model['main_model'])
        & main['backbone'].astype(str).eq(model['main_backbone'])
        & main['probe_name'].astype(str).eq('temporal_attn')
    ].copy()
    if subset.empty:
        return {'status': 'missing main seed42 row'}

    subset['_test'] = pd.to_numeric(subset['test_primary_metric'], errors='coerce')
    subset['_selection'] = subset['_test']
    subset = subset.dropna(subset=['_selection', '_test'])
    if subset.empty:
        return {'status': 'missing main seed42 metrics'}

    row = subset.sort_values('_selection', ascending=False, kind='stable').iloc[0]
    layer = norm_layer(row.get('selected_slot')) or norm_layer(row.get('probe_layer')) or norm_layer(row.get('excel_layer'))
    layer_label = norm_layer(row.get('layer_label')) or norm_layer(row.get('excel_layer')) or layer
    mapped_layer = LTX_LABEL_TO_SLOT.get(layer, layer)
    mapped_layer = LTX_LABEL_TO_SLOT.get(layer_label, mapped_layer)
    return {
        'layer': mapped_layer,
        'layer_label': layer_label,
        'display_layer': format_layer_display(model, mapped_layer, layer_label),
        'train': as_float(row.get('train_primary_metric')),
        'val': as_float(row.get('val_primary_metric')),
        'test': as_float(row.get('test_primary_metric')),
        'test_acc': as_float(row.get('test_accuracy')),
        'source_status': str(row.get('config_status') or ''),
        'source': str(row.get('config_source') or row.get('excel_workbook') or ''),
        'status': 'done',
    }


def load_paper_row(dataset: dict, model: dict) -> dict[str, str]:
    main = main_seed42_row(dataset, model)
    base = {
        'Dataset': dataset['label'],
        'Model': model['label'],
        'Layer': 'NA',
        'Main seed42 primary test': 'NA',
        'Random-label primary test': 'NA',
        'Primary drop pp': 'NA',
        'Main seed42 accuracy test': 'NA',
        'Random-label accuracy test': 'NA',
        'Accuracy drop pp': 'NA',
    }
    if main.get('status') != 'done':
        return base

    random = random_layer_metrics(dataset, model, main['layer'], main['layer_label'])
    main_test = as_float(main.get('test'))
    random_test = as_float(random.get('test'))
    primary_drop = None if main_test is None or random_test is None else random_test - main_test
    main_acc = as_float(main.get('test_acc'))
    random_acc = as_float(random.get('test_acc'))
    accuracy_drop = None if main_acc is None or random_acc is None else random_acc - main_acc
    return {
        **base,
        'Layer': str(main['display_layer']),
        'Main seed42 primary test': fmt(main_test),
        'Random-label primary test': fmt(random_test),
        'Primary drop pp': fmt(primary_drop),
        'Main seed42 accuracy test': fmt(main_acc),
        'Random-label accuracy test': fmt(random_acc),
        'Accuracy drop pp': fmt(accuracy_drop),
    }


def make_table() -> list[dict[str, str]]:
    return [load_control_row(dataset, model) for dataset in DATASETS for model in MODELS]


def make_paper_table() -> list[dict[str, str]]:
    return [load_paper_row(dataset, model) for dataset in DATASETS for model in MODELS]


In [3]:
paper_rows = make_paper_table()
table_rows = make_table()

if pd is not None:
    paper_table = pd.DataFrame(paper_rows)
    audit_table = pd.DataFrame(table_rows)
    display(Markdown('**Paper table: main seed42 vs random labels at the main test-selected layer**'))
    display(paper_table)
    display(Markdown('**Audit table: random-label best layer selected within the control run**'))
    display(audit_table)
else:
    for title, rows in [('Paper table', paper_rows), ('Audit table', table_rows)]:
        print(title)
        columns = list(rows[0])
        print('| ' + ' | '.join(columns) + ' |')
        print('| ' + ' | '.join(['---'] * len(columns)) + ' |')
        for row in rows:
            print('| ' + ' | '.join(str(row[col]) for col in columns) + ' |')


**Paper table: main seed42 vs random labels at the main test-selected layer**

,Dataset,Model,Layer,Main seed42 primary test,Random-label primary test,Primary drop pp,Main seed42 accuracy test,Random-label accuracy test,Accuracy drop pp
0,IntPhys2,V-JEPA,16,66.67,13.73,-52.94,78.92,51.96,-26.96
1,IntPhys2,V-JEPA 2,40,78.43,11.76,-66.67,82.84,50.00,-32.84
2,IntPhys2,V-JEPA 2.1,48,60.78,19.61,-41.18,72.55,44.12,-28.43
3,IntPhys2,VideoMAE,24,76.47,13.73,-62.75,80.88,50.49,-30.39
4,IntPhys2,VideoMAE-v2,20,62.75,7.84,-54.90,72.55,50.00,-22.55
5,IntPhys2,LTX-Video,48 (noise 0.2),43.14,9.80,-33.33,64.22,48.53,-15.69
6,MVP,V-JEPA,24,95.05,10.11,-84.93,97.27,50.61,-46.66
7,MVP,V-JEPA 2,40,94.03,8.80,-85.24,96.51,49.24,-47.27
8,MVP,V-JEPA 2.1,38,94.94,16.99,-77.96,97.12,49.04,-48.08
9,MVP,VideoMAE,24,92.01,13.45,-78.56,95.10,50.05,-45.05


**Audit table: random-label best layer selected within the control run**

,Dataset,Model,Best layer,LR,Train VOE/PC,Train acc,Val VOE/PC,Val acc,Test VOE/PC,Test acc,Status
0,IntPhys2,V-JEPA,8,5e-4,18.54,53.31,29.41,57.84,9.80,50.49,done
1,IntPhys2,V-JEPA 2,20,5e-5,23.84,50.17,29.41,51.96,11.76,50.49,done
2,IntPhys2,V-JEPA 2.1,12,5e-4,19.21,50.00,27.45,53.43,9.80,47.06,done
3,IntPhys2,VideoMAE,32,1e-5,34.44,51.66,29.41,54.41,13.73,50.49,done
4,IntPhys2,VideoMAE-v2,30,5e-4,16.56,50.00,25.49,53.92,19.61,52.94,done
5,IntPhys2,LTX-Video,24 (noise 0.1),5e-5,23.18,50.50,29.41,52.94,17.65,50.49,done
6,MVP,V-JEPA,8,5e-4,12.28,49.34,14.05,50.61,13.85,51.06,done
7,MVP,V-JEPA 2,30,5e-4,15.68,50.24,18.00,51.82,16.48,51.77,done
8,MVP,V-JEPA 2.1,38,1e-4,18.28,50.99,18.81,50.96,16.99,49.04,done
9,MVP,VideoMAE,24,1e-5,18.15,53.69,14.56,51.31,13.45,50.05,done


In [4]:
paper_out_path = RESULTS_DIR / 'random_label_controls_main_vs_seed42.csv'
audit_out_path = RESULTS_DIR / 'random_label_controls_table.csv'
for out_path, rows in [(paper_out_path, paper_rows), (audit_out_path, table_rows)]:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
paper_out_path, audit_out_path


(PosixPath('/gpfs/home2/spunzo1/Probe4Physics/results/random_label_controls_main_vs_seed42.csv'),
 PosixPath('/gpfs/home2/spunzo1/Probe4Physics/results/random_label_controls_table.csv'))